# ARD-VAE reproduction (MNIST, L = 16)
Paper: *ARD-VAE: A Statistical Formulation to Find the Relevant Latent Dimensions of Variational Autoencoders* (WACV).

This notebook reproduces **Table 2 (MNIST column)** and **Fig. 1** of the paper:
- baseline VAE (fixed N(0, I) prior) vs ARD-VAE (hierarchical prior, Eq. 8-9, 16-17)
- number of ACTIVE latent dimensions (relevance score, Eq. 18-20, 99% variance rule)
- FID of generated samples

**Run order:** Runtime > Change runtime type > GPU (T4), then Run all.

In [ ]:
!pip -q install torchmetrics torch-fidelity

In [ ]:
import time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets
from torchmetrics.image.fid import FrechetInceptionDistance

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# ---------------- configuration ----------------
L        = 16        # initial latent size (paper: MNIST L=16)
EPOCHS   = 50
BS       = 100       # minibatch size (paper uses 100 for Jacobian batch)
LR       = 1e-3
BETA     = 1.0       # KL weight; paper tunes it to reach a target reconstruction loss
N_ALPHA  = 10000     # |X_alpha| used to estimate the hyperprior (paper: 10K)
WARMUP   = 1         # epochs with fixed N(0,I) prior before D_z exists (our assumption)
SEEDS    = [0, 1, 2, 3, 4]   # paper uses 5 seeds
N_FID    = 10000     # generated / real images for FID
VAR_THR  = 0.99      # percentage-of-variance rule for ACTIVE dims

In [ ]:
# ---------------- data (kept on GPU as tensors) ----------------
tr = datasets.MNIST('/content/data', train=True,  download=True)
te = datasets.MNIST('/content/data', train=False, download=True)
Xtr = tr.data.float().div(255).unsqueeze(1).to(device)   # 60000 x 1 x 28 x 28
Xte = te.data.float().div(255).unsqueeze(1).to(device)
Ntrain = Xtr.shape[0]
print(Xtr.shape, Xte.shape)

In [ ]:
# ---------------- model ----------------
class VAE(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.L = L
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 256), nn.ReLU(), nn.Linear(256, 2 * L))
        self.dec_fc = nn.Sequential(
            nn.Linear(L, 256), nn.ReLU(), nn.Linear(256, 64 * 7 * 7), nn.ReLU())
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1))
    def encode(self, x):
        mu, logvar = self.enc(x).chunk(2, 1)
        return mu, logvar
    def decode_logits(self, z):
        return self.dec_conv(self.dec_fc(z).view(-1, 64, 7, 7))
    def decode(self, z):
        return torch.sigmoid(self.decode_logits(z))

# ---------------- ARD hyperprior (Eq. 8, 9, 18) ----------------
@torch.no_grad()
def estimate_prior(m, X):
    """D_z = z sampled from encoder posterior on X_alpha.
    With a0 = b0 = 0 and mu_alpha = 0:  a = n/2,  b = sum(z^2)/2,  sigma_hat^2 = b/a."""
    m.eval(); zs = []
    for i in range(0, len(X), 1000):
        mu, lv = m.encode(X[i:i + 1000])
        zs.append(mu + torch.randn_like(mu) * (0.5 * lv).exp())
    z = torch.cat(zs); n = z.shape[0]
    a = torch.full((m.L,), n / 2.0, device=z.device)
    b = 0.5 * (z ** 2).sum(0)
    m.train()
    return (b / a).clamp_min(1e-8)

# ---------------- relevance score (Eq. 19, 20) ----------------
def jacobian_weights(m, X, n_batches=10, bs=100):
    """w_l = mean_i sum_k (d xhat_k / d mu_l)^2 via forward-mode JVP (one pass per latent axis)."""
    m.eval(); w = torch.zeros(m.L, device=X.device)
    for b in range(n_batches):
        x = X[b * bs:(b + 1) * bs]
        with torch.no_grad():
            mu, _ = m.encode(x)
        for l in range(m.L):
            t = torch.zeros_like(mu); t[:, l] = 1.0
            _, jv = torch.func.jvp(m.decode, (mu,), (t,))
            w[l] += (jv.detach() ** 2).flatten(1).sum(1).mean()
    m.train()
    return w / n_batches

def n_active(score, thr=VAR_THR):
    s, _ = torch.sort(score, descending=True)
    c = torch.cumsum(s, 0) / s.sum()
    return int((c < thr).sum().item() + 1)

In [ ]:
# ---------------- FID (real features computed once) ----------------
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)
prep = lambda x: x.clamp(0, 1).repeat(1, 3, 1, 1)
for i in range(0, N_FID, 500):
    fid_metric.update(prep(Xte[i:i + 500]), real=True)

@torch.no_grad()
def fid_of(m, s2, seed):
    m.eval()
    g = torch.Generator(device=device).manual_seed(1000 + seed)
    for i in range(0, N_FID, 500):
        z = torch.randn(500, m.L, device=device, generator=g) * s2.sqrt()
        fid_metric.update(prep(m.decode(z)), real=False)
    f = fid_metric.compute().item()
    fid_metric.reset()          # keeps real statistics
    m.train()
    return f

In [ ]:
# ---------------- training ----------------
def run(method, seed, L=L, epochs=EPOCHS, beta=BETA, verbose=True):
    """method: 'vae' (fixed N(0,I) prior) or 'ard' (hierarchical prior, Eq. 16-17)."""
    torch.manual_seed(seed); np.random.seed(seed)
    m = VAE(L).to(device); opt = torch.optim.Adam(m.parameters(), lr=LR)
    s2 = torch.ones(L, device=device)
    hist = {'min': [], 'max': [], 'loss': []}
    t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(Ntrain, device=device)
        idx_alpha, idx_sgd = perm[:N_ALPHA], perm[N_ALPHA:]          # X_alpha / X_sgd split, refreshed each epoch
        if method == 'ard' and ep >= WARMUP:
            s2 = estimate_prior(m, Xtr[idx_alpha])                   # D_z updated once per epoch (lagged)
        tot = 0.0; nb = 0
        for i in range(0, len(idx_sgd) - BS + 1, BS):
            x = Xtr[idx_sgd[i:i + BS]]
            mu, lv = m.encode(x)
            z = mu + torch.randn_like(mu) * (0.5 * lv).exp()
            rec = F.binary_cross_entropy_with_logits(m.decode_logits(z), x, reduction='none').flatten(1).sum(1)
            # Eq. 17 (Gaussian approximation of the Student-t prior); s2 = 1 gives the standard VAE KL
            kl = 0.5 * (-m.L - lv.sum(1) + torch.log(s2).sum() + ((mu ** 2 + lv.exp()) / s2).sum(1))
            loss = (rec + beta * kl).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        hist['loss'].append(tot / nb)
        if method == 'ard':
            hist['min'].append(s2.min().item()); hist['max'].append(s2.max().item())
        if verbose and (ep % 10 == 0 or ep == epochs - 1):
            print(f'  [{method} seed{seed}] ep {ep:3d} loss {tot/nb:.2f}  ({time.time()-t0:.0f}s)')

    # final prior estimate with a fresh X_alpha
    if method == 'ard':
        s2 = estimate_prior(m, Xtr[torch.randperm(Ntrain, device=device)[:N_ALPHA]])
    else:
        s2 = torch.ones(L, device=device)
    w = jacobian_weights(m, Xtr[:1000])
    score = w * s2                                                   # Eq. 20
    with torch.no_grad():                                            # test reconstruction (BCE, z = mu)
        m.eval(); mu, _ = m.encode(Xte[:5000])
        rec_te = F.binary_cross_entropy_with_logits(m.decode_logits(mu), Xte[:5000], reduction='none').flatten(1).sum(1).mean().item()
        m.train()
    return dict(method=method, seed=seed, L=L, active=n_active(score), fid=fid_of(m, s2, seed),
                rec_test=rec_te, s2=s2.cpu().numpy(), score=score.cpu().numpy(), hist=hist, model=m)

In [ ]:
# ---------------- main experiment: VAE vs ARD-VAE on MNIST (L=16) ----------------
results = {'vae': [], 'ard': []}
for method in ['vae', 'ard']:
    for seed in SEEDS:
        r = run(method, seed)
        results[method].append(r)
        print(f'>> {method} seed {seed}: active={r["active"]}  FID={r["fid"]:.2f}  test-rec={r["rec_test"]:.2f}')

In [ ]:
# ---------------- comparison table: paper vs reproduced ----------------
ms = lambda v: f'{np.mean(v):.2f} ± {np.std(v):.2f}'
df = pd.DataFrame([
    ['VAE',     '16',           '28.78 ± 0.48', ms([r['active'] for r in results['vae']]), ms([r['fid'] for r in results['vae']])],
    ['ARD-VAE', '12.80 ± 0.40', '22.24 ± 0.57', ms([r['active'] for r in results['ard']]), ms([r['fid'] for r in results['ard']])],
], columns=['Method', 'ACTIVE (paper)', 'FID (paper)', 'ACTIVE (ours)', 'FID (ours)'])
print(df.to_string(index=False))
print('\nMean test reconstruction loss (BCE):',
      {k: round(float(np.mean([r['rec_test'] for r in v])), 2) for k, v in results.items()})
df.to_csv('comparison_table.csv', index=False)

In [ ]:
# ---------------- Fig. 1-style plots (ARD-VAE, seed 0) + sample grid ----------------
r = results['ard'][0]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(np.sort(r['s2'])[::-1], 'o-'); ax[0].set_title('Estimated variance (Eq. 18), sorted'); ax[0].set_xlabel('latent dim'); ax[0].set_ylabel('variance')
ax[1].plot(np.sort(r['score'])[::-1], 'o-'); ax[1].set_title('Relevance score (Eq. 20), sorted'); ax[1].set_xlabel('latent dim')
ax[2].plot(r['hist']['max'], label='max'); ax[2].plot(r['hist']['min'], label='min'); ax[2].legend()
ax[2].set_title('Min / max estimated variance during training'); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.savefig('fig_ard_variances.png', dpi=150); plt.show()

def grid(m, s2, n=10):
    m.eval()
    with torch.no_grad():
        x = m.decode(torch.randn(n * n, m.L, device=device) * s2.sqrt()).cpu().view(n, n, 28, 28)
    m.train()
    return x.permute(0, 2, 1, 3).reshape(n * 28, n * 28).numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(grid(results['vae'][0]['model'], torch.ones(L, device=device)), cmap='gray'); ax[0].set_title('VAE samples'); ax[0].axis('off')
ax[1].imshow(grid(r['model'], torch.tensor(r['s2'], device=device)), cmap='gray'); ax[1].set_title('ARD-VAE samples'); ax[1].axis('off')
plt.tight_layout(); plt.savefig('fig_samples.png', dpi=150); plt.show()

In [ ]:
# ---------------- OPTIONAL: Fig. 1a with several initial latent sizes (1 seed each) ----------------
# Shows that the relevance scores are nearly identical across L. Takes a few extra minutes.
RUN_L_SWEEP = False
if RUN_L_SWEEP:
    sw = {Lx: run('ard', 0, L=Lx, verbose=False) for Lx in [16, 32, 64, 128]}
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for Lx, rr in sw.items():
        ax[0].plot(np.sort(rr['s2'])[::-1], label=f'L={Lx}'); ax[1].plot(np.sort(rr['score'])[::-1], label=f'L={Lx}')
        print(f'L={Lx}: active={rr["active"]}  FID={rr["fid"]:.2f}')
    ax[0].set_title('Estimated variances'); ax[1].set_title('Relevance scores'); ax[0].legend(); ax[1].legend()
    plt.tight_layout(); plt.savefig('fig_L_sweep.png', dpi=150); plt.show()

In [ ]:
# ---------------- OPTIONAL: beta sensitivity (paper tunes beta to a target recon loss) ----------------
RUN_BETA_SWEEP = False
if RUN_BETA_SWEEP:
    for b in [0.5, 1.0, 2.0]:
        rr = run('ard', 0, beta=b, verbose=False)
        print(f'beta={b}: active={rr["active"]}  FID={rr["fid"]:.2f}  test-rec={rr["rec_test"]:.2f}')